#### LAB 08 — Pretrained Transformers and BERT Fine-Tuning



In [1]:
import sys, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display
LAB_DIR = Path.cwd() / 'lab8' if (Path.cwd() / 'lab8' / 'pretrained_classification.py').is_file() else Path.cwd()
sys.path.insert(0, str(LAB_DIR.resolve()))
import pretrained_classification as pc
pc.configure()
tokenizer = pc.load_tokenizer()
print('Model:', pc.MODEL_ID, '| revision:', pc.REVISION)
print('Tokenizer:', type(tokenizer).__name__, '| vocab:', tokenizer.vocab_size)
print('Device:', pc.DEVICE, '| torch:', torch.__version__, '| threads:', torch.get_num_threads())

Model: prajjwal1/bert-tiny | revision: 6f75de8b60a9f8a2fdf7b69cbd86d9e64bcb3837
Tokenizer: BertTokenizer | vocab: 30522
Device: cpu | torch: 2.11.0+cpu | threads: 4


W1111 21:49:01.751000 12760 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


#### 1. Pretraining vs Fine-tuning

| Cách học | Khởi tạo encoder | Dữ liệu/mục tiêu | Tham số cập nhật |
|---|---|---|---|
| From scratch | Ngẫu nhiên | IMDb, sentiment loss | Encoder + head |
| Pretraining | Ban đầu ngẫu nhiên, sau đó học trên corpus lớn | Ví dụ MLM: dự đoán token bị che | Các tham số phục vụ pretraining |
| Feature extraction | Pretrained | IMDb, sentiment loss | Chỉ head |
| Fine-tuning | Pretrained | IMDb, sentiment loss | Encoder + head |


#### 2. Subword Tokenization

`Raw text → WordPiece tokenizer → IDs → Embedding → Transformer`. 

Một từ có thể giữ nguyên hoặc tách thành nhiều mảnh tùy vocabulary. BERT uncased chuẩn hóa chữ hoa/thường; tiền tố `##` biểu thị mảnh tiếp nối từ. Cách tách cụ thể được xác định bằng kết quả chạy tokenizer.

In [2]:
subword_rows = []
for word in ['unbelievable', 'tokenization', 'cats', 'playing', 'microarchitectures']:
    pieces = tokenizer.tokenize(word)
    subword_rows.append({'Word': word, 'WordPiece tokens': pieces, 'IDs': tokenizer.convert_tokens_to_ids(pieces), 'Count': len(pieces)})
display(pd.DataFrame(subword_rows))

,Word,WordPiece tokens,IDs,Count
0,unbelievable,[unbelievable],[23653],1
1,tokenization,"[token, ##ization]","[19204, 3989]",2
2,cats,[cats],[8870],1
3,playing,[playing],[2652],1
4,microarchitectures,"[micro, ##ar, ##chi, ##tec, ##tures]","[12702, 2906, 5428, 26557, 22662]",5


#### 3. Special Tokens

| Token | Vai trò |
|---|---|
| [CLS] | Token đầu; hidden state có ngữ cảnh của nó được đưa vào head classification. |
| [SEP] | Kết thúc một sequence, phân cách hai sequence khi nhập cặp câu. |
| [PAD] | Lấp đầy batch đến độ dài chung; không phải nội dung thật. |
| [MASK] | Thay token cần dự đoán trong MLM. |
| [UNK] | Đơn vị tokenizer không biểu diễn được bằng vocabulary. |

Với một câu: `[CLS] A [SEP]`; cặp câu: `[CLS] A [SEP] B [SEP]`. [CLS] không tự mang một nhãn sentiment; encoder và head phải được huấn luyện phù hợp.

#### 4. Attention Mask

Mask bằng 1 ở token hợp lệ, 0 ở PAD; special tokens [CLS]/[SEP] cũng có mask 1. Mask ngăn PAD được dùng như key nội dung. Hidden state tại một vị trí PAD không nhất thiết là vector 0; head của bài lấy [CLS], không lấy trung bình bao gồm PAD.

Ví dụ `[CLS] the cat eats fish [PAD] [PAD]` trong đề có mask `[1,1,1,1,1,0,0]`. Tokenizer BERT thực tế còn tự thêm [SEP] như output mục 10.

#### 5. Masked Language Modeling

`The cat [MASK] fish.` → dự đoán token bị che bằng ngữ cảnh hai phía. Với tập vị trí được chọn $M$:

$$L=-\sum_{i\in M}\log P(y_i\mid x_{masked}).$$

Chỉ các vị trí mục tiêu đóng góp MLM loss; thường lấy trung bình loss để so sánh batch. MLM dự đoán token trong vocabulary, còn sentiment classification dự đoán hai nhãn toàn review. Bài này sử dụng checkpoint đã pretrained, không chạy lại pretraining.

#### 10. Calculation Exercise 2 — Tokenization

Tokenize câu **“The cats are playing.”** bằng tokenizer của checkpoint, padding đến 12 vị trí để quan sát attention mask. Bảng kết quả phân biệt từ, mảnh WordPiece, dấu câu, special tokens và PAD.

In [3]:
sentence = 'The cats are playing.'
encoded_sentence = tokenizer(sentence, padding='max_length', max_length=12, truncation=True, return_tensors='pt')
sentence_tokens = tokenizer.convert_ids_to_tokens(encoded_sentence['input_ids'][0].tolist())
tokenization_table = pd.DataFrame({'Token': sentence_tokens, 'ID': encoded_sentence['input_ids'][0].tolist(),
                                   'Attention mask': encoded_sentence['attention_mask'][0].tolist(),
                                   'Token type ID': encoded_sentence['token_type_ids'][0].tolist()})
display(tokenization_table)
word_encoding = tokenizer(['The', 'cats', 'are', 'playing'], is_split_into_words=True, add_special_tokens=False)
token_counts = {'Words': 4, 'WordPiece pieces for words': len(word_encoding['input_ids']),
                'Continuation pieces (##)': sum(t.startswith('##') for t in sentence_tokens),
                'Content tokens including punctuation': len(tokenizer.tokenize(sentence)),
                'Special tokens without PAD': tokenizer.num_special_tokens_to_add(pair=False),
                'Non-PAD tokens': int(encoded_sentence['attention_mask'].sum()),
                'PAD tokens': int((encoded_sentence['attention_mask']==0).sum())}
display(pd.DataFrame(token_counts.items(), columns=['Quantity', 'Count']))

,Token,ID,Attention mask,Token type ID
0,[CLS],101,1,0
1,the,1996,1,0
2,cats,8870,1,0
3,are,2024,1,0
4,playing,2652,1,0
5,.,1012,1,0
6,[SEP],102,1,0
7,[PAD],0,0,0
8,[PAD],0,0,0
9,[PAD],0,0,0


,Quantity,Count
0,Words,4
1,WordPiece pieces for words,4
2,Continuation pieces (##),0
3,Content tokens including punctuation,5
4,Special tokens without PAD,2
5,Non-PAD tokens,7
6,PAD tokens,5


Kết quả: `[CLS], the, cats, are, playing, ., [SEP]` có IDs `[101,1996,8870,2024,2652,1012,102]`; thêm 5 `[PAD]` có ID 0. Mask là `[1,1,1,1,1,1,1,0,0,0,0,0]`.

Câu có **4 từ**, mỗi từ ứng với 1 mảnh WordPiece trong vocabulary này, **0 mảnh tiếp nối `##`**. Có 5 content tokens khi tính cả dấu chấm, 7 token không PAD khi tính [CLS]/[SEP], và 12 vị trí sau padding. “Subword tokenizer” không có nghĩa mọi từ đều bắt buộc bị tách nhỏ.

#### 12. Core Implementation — Tokenization

`input_ids` là chỉ số trong vocabulary, không phải trọng số attention. `attention_mask` đánh dấu nội dung hợp lệ/PAD. BERT còn trả `token_type_ids` để phân biệt hai segment; một câu có segment 0. `max_length` bao gồm cả special tokens; truncation bên phải cắt phần đuôi trước khi thêm [SEP].

In [4]:
tokens = tokenizer(['This movie is great.', 'I did not enjoy this slow and repetitive movie.'],
                   padding=True, truncation=True, max_length=32, return_tensors='pt')
print('tokens.keys():', list(tokens.keys()))
for key, value in tokens.items():
    print(key, tuple(value.shape))
    print(value)
for ids in tokens['input_ids']:
    print(tokenizer.convert_ids_to_tokens(ids.tolist()))

tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']
input_ids (2, 12)
tensor([[  101,  2023,  3185,  2003,  2307,  1012,   102,     0,     0,     0,
             0,     0],
        [  101,  1045,  2106,  2025,  5959,  2023,  4030,  1998, 23563,  3185,
          1012,   102]])
token_type_ids (2, 12)
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask (2, 12)
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
['[CLS]', 'this', 'movie', 'is', 'great', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
['[CLS]', 'i', 'did', 'not', 'enjoy', 'this', 'slow', 'and', 'repetitive', 'movie', '.', '[SEP]']


Phần tokenization xác định token sequence, IDs, special tokens và attention mask. Các thí nghiệm phân loại bắt đầu từ mục 13 trong `fine_tuning.ipynb`. Hai notebook có thể chạy độc lập.
